# §49: **K5fepre**, **K5fepost** (edge normal windows, once per epoch) and **K5fbk** (bottom-k MIL) on K5f, s2024

Rule: `core/docs/final/KIM_RUN.md` **§49**, written before any number of these rows. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. `nexar_test` is never touched. **K5f is the control and is not retrained; nothing is adopted at n = 1.**

* **K5fepre / K5fepost**: one 20-row normal window right before / right after each positive train video's span (gap 0), drawn **exactly once per epoch** (~449 of the 1,689 normal slots); the other slots draw K5f's items. Config diff to K5f = 0; only the corpus differs.
* **K5fbk**: `loss.bottomk_weight=1.0`, `loss.bottomk_topk_pct=5` (k = 4) on K5f's corpus.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight (K5f run + reads, bakes, census) | no | 2-5 min |
| 4 | pytest on the touched code — **raises if red** | no | 2-3 min |
| 5 | build the two edge corpora + their asserts | no | 10-20 min |
| 6 | train the rows of `RUN_ROWS` + ED6 checks | yes | 1 × K5f per row |
| 7 | zero-shot reads: DoTA-CAP-dev (protocol B), DADA whole, TAD, DAD | yes | 20-30 min per row |
| 8 | `nexar_val` read with `--save-scores` | yes | 10-20 min per row |
| 9 | report: verdicts, four-set table, pre-onset -> **paste `REPORTS/k5_edge_summary.md` to Claude** | no | 3-5 min |

`RUN_ROWS` (cell 2) may be shortened to spread the rows over sessions; a re-run skips what is on Drive, and step 9 reports every row that is complete. Thesis and Thesis-V2 are only read; everything new goes to `Thesis-Final/outputs/kim_edge/` and `Thesis-Final/data/Nexar_window_edge{pre,post}/`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, Nexar caches, census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5 and its reads, DAD; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 49. K5fepre' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §49 -- re-upload kat-vad/'
assert 'def build_window_edge' in (REPO / 'core/tools/nexar_build.py').read_text(), 'nexar_build has no window_edge* -- re-upload kat-vad/'
assert 'def _normal_anchor' in (REPO / 'core/data/dataset.py').read_text(), 'dataset.py has no forced normals (§49) -- re-upload kat-vad/: the edge rows would train as K5f'
assert (REPO / 'core/tools/k5_edge_report.py').is_file(), 'no k5_edge_report -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/ (training now would reproduce K5)'
assert '--save-scores' in (REPO / 'core/tools/nexar_eval.py').read_text(), 'nexar_eval has no --save-scores -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                  # sub-folder name of every eval read (eval/<set>/K5)
K5F      = constants.KIM_FIXED_ARM                 # the control (§31 reference)
PRE, POST, BK = constants.KIM_EDGE_ROWS            # K5fepre, K5fepost, K5fbk (§49)
RUN_ROWS = [PRE, POST, BK]                         # shorten to split the rows over sessions
SEED     = constants.KIM_SEED                      # 2024
ENC      = constants.V5_ENCODER                    # V2-S
DATASET  = constants.NEXAR_DATASET
C_WINDOW = constants.NEXAR_CONSTRUCTION_WINDOW
C_EDGE   = {PRE: constants.NEXAR_CONSTRUCTION_WINDOW_EDGEPRE, POST: constants.NEXAR_CONSTRUCTION_WINDOW_EDGEPOST}
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL         # nexar_test is never read here
TAD_NAME = constants.TAD_DATASET
DAD_NAME, DAD_STRIDE = constants.DAD_DATASET, constants.DAD_STRIDE

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C_WINDOW}'     # K5's corpus: the reference of every assert
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                # nexar_train statistics, every train video
# K5f: the control, its run and its reads (§31)
FIXED    = constants.OUTPUT_ROOT / constants.KIM_FIXED_OUTPUT_DIRNAME
K5F_RUN  = FIXED / K5F / K5 / f's{SEED}'
K5F_EV   = FIXED / K5F / 'eval'
# Zero-shot inputs: the bakes K5 was read on
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
A2_TAD   = V2_CACHE / 'v2' / 'A2_S' / f'{TAD_NAME}_s8_nexar'
A2_DAD   = constants.CACHE_ROOT / 'v2' / 'A2_S' / f'{DAD_NAME}_s{DAD_STRIDE}_nexar'
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
TAD      = constants.DATA_ROOT / TAD_NAME
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
# Nexar
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
# Outputs: one folder per row, each read like K5 (eval/<set>/K5, eval/nexar/<row>)
ROOT     = constants.OUTPUT_ROOT / constants.KIM_EDGE_OUTPUT_DIRNAME
DATA     = {row: Path(DRIVE_F) / 'data' / f'{DATASET}_{c}' for row, c in C_EDGE.items()} | {BK: WINDOW}  # edge corpora: step 5
BUILD    = ROOT / 'REPORTS' / 'build_report_edge.json'
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/k5edge')
LOCAL.mkdir(parents=True, exist_ok=True)


def row_paths(row):
    run_dir = ROOT / row / K5 / f's{SEED}'
    return run_dir, run_dir / 'checkpoint_last.pt', ROOT / row / 'eval'


INPUTS = (('Nexar_window labels', WINDOW / constants.LABELS_TRAIN_FILENAME), ('Nexar A2 bake', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          ('K5f checkpoint', K5F_RUN / 'checkpoint_last.pt'), ('K5f verified', K5F_RUN / 'verified.json'),
          ('K5f config', K5F_RUN / 'config.yaml'), ('K5f nexar_val read', K5F_EV / 'nexar' / K5F / 'video_stats.json'), ('K5f nexar_val scores', K5F_EV / 'nexar' / K5F / 'scores'),
          ('K5f metrics', K5F_RUN / 'metrics.jsonl'),
          ('DADA A2 bake', A2_DADA), ('DoTA-CAP A2 bake', A2_DOTA), ('TAD A2 bake', A2_TAD), ('DAD A2 bake', A2_DAD),
          ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
          ('DoTA s1', DOTA_S1), ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME),
          ('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U),
          ('K5f DoTA read', K5F_EV / 'dota' / K5), ('K5f DADA read', K5F_EV / 'dada' / K5),
          ('K5f TAD read', K5F_EV / 'tad' / K5 / f's{SEED}' / 'scores'),
          ('K5f DAD read', K5F_EV / constants.DAD_OUTPUT_SET / K5 / f's{SEED}' / 'results.json'))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
assert set(RUN_ROWS) <= {PRE, POST, BK}, RUN_ROWS
print(f'{len(INPUTS)} inputs found | rows {RUN_ROWS} | runs/reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 6-8 need one'
df -h /content | tail -1

In [ ]:
import torch
import yaml

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(drive_dir, steps):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())



# Preflight on small files only (C39)
k5f_cfg = _flat(yaml.safe_load((K5F_RUN / 'config.yaml').read_text()))
assert k5f_cfg['kim.enabled'] and k5f_cfg['kim.prototypes'] and k5f_cfg['kim.edge_mask'], 'K5F_RUN is not K5f'
assert k5f_cfg['dvs.theta'] == 1.0 and k5f_cfg['data.dataset'] == DATASET, 'K5F_RUN is not K5f on Nexar'
STEPS = json.loads((K5F_RUN / 'verified.json').read_text())['step']            # 1,749
EPOCHS = k5f_cfg['train.num_epochs']                                           # 33
assert is_verified(K5F_RUN, STEPS) and metrics_last(K5F_RUN / 'metrics.jsonl')['global_step'] == STEPS, 'K5f: J10'
SHA = manifest(A2_NEXAR)['train_ids_sha1']
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA), ('DoTA-CAP', A2_DOTA), ('TAD', A2_TAD), ('DAD', A2_DAD)):
    m = manifest(d)
    assert (m['crn'], m['motion'], m['train_ids_sha1']) == (constants.V2_OFF, ENC, SHA), f'{name} bake: {m}'
census = json.loads(CENSUS.read_text())['videos']
TRAIN_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_TRAIN))
NEG_IDS = [v for v in TRAIN_IDS if census[v]['label'] == 0]
DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert (len(TRAIN_IDS), len(NEG_IDS)) == (900, 449), (len(TRAIN_IDS), len(NEG_IDS))
assert len(DADA_IDS) == 370 and len(TAD_IDS) == constants.V2_TADZ_VIDEOS and len(DAD_IDS) == constants.DAD_TEST_VIDEOS
for name, d, ids in (('Nexar train bake', A2_NEXAR, TRAIN_IDS), ('DADA bake', A2_DADA, DADA_IDS),
                     ('DoTA-CAP bake', A2_DOTA, CAP_IDS), ('TAD bake', A2_TAD, TAD_IDS), ('DAD bake', A2_DAD, DAD_IDS),
                     ('Nexar CLIP', NEXAR_X, NEXAR_IDS), ('Nexar V2-S', NEXAR_U, NEXAR_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
print(f'preflight OK: K5f step {STEPS} ({EPOCHS} epochs) | train {len(TRAIN_IDS)} ({len(NEG_IDS)} negative) | '
      f'DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)}, TAD {len(TAD_IDS)}, DAD {len(DAD_IDS)}, {NEXAR_SPLIT} {len(NEXAR_IDS)}')


## 4. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**


In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_nexar_pipeline.py',
     'core/tests/test_k5_edge_report.py', 'core/tests/test_dvs_dataset.py', 'core/tests/test_pre_onset_read.py',
     'core/tests/test_baseline_losses.py'])


## 5. Build `Nexar_window_edgepre` / `_edgepost` (§49 ED2, ED4; header reads of the s1 cache only)

Asserts: `labels_train`, `frame_labels_test`, `defs`, `meta` identical to `Nexar_window`; every `Nexar_window` window unchanged; one forced window per video, 20 rows, gap 0, all normal, from a positive train video; step plan = K5f's.


In [ ]:
need = [row for row in (PRE, POST) if row in RUN_ROWS and not (DATA[row] / constants.FORCED_NORMALS_FILENAME).exists()]
if need:
    local = LOCAL / 'corpus'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.nexar_build', 'corpus', '--census', CENSUS, '--clip-s1-dir', NEXAR_X,
         '--out-root', local, '--constructions', C_EDGE[PRE], C_EDGE[POST], '--report', LOCAL / 'build_report.json'], log=LOCAL / 'build.log')
    for row in (PRE, POST):
        publish(local / DATA[row].name, DATA[row])   # forced_normals.json may land before windows.json: both checked below
    publish(LOCAL / 'build_report.json', BUILD)
    flush_drive()

win = json.loads((WINDOW / constants.LABELS_TRAIN_FILENAME).read_text())
base_windows = json.loads((WINDOW / constants.WINDOWS_FILENAME).read_text())
POS_IDS = {base_windows[w]['source'] for w, l in win.items() if l == 1}
assert (sum(win.values()), len(win) - sum(win.values()), len(POS_IDS)) == (1689, 5622, 451), 'Nexar_window is not K5f\'s corpus'
assert not (WINDOW / constants.FORCED_NORMALS_FILENAME).exists(), 'Nexar_window itself holds forced normals: K5fbk would not be K5f + bottom-k'
LENGTH = constants.NEXAR_WINDOW_LENGTH
for row in (r for r in (PRE, POST) if r in RUN_ROWS):
    d = DATA[row]
    build = json.loads(BUILD.read_text())
    info, plan = build['constructions'][C_EDGE[row]], build['plan'][C_EDGE[row]]
    assert (plan['steps'], plan['epochs']) == (STEPS, EPOCHS), f'{row}: step plan {plan} != K5f'
    for name in (constants.LABELS_TRAIN_FILENAME, constants.FRAME_LABELS_TEST_FILENAME, constants.DEFS_FILENAME, constants.META_FILENAME):
        assert json.loads((d / name).read_text()) == json.loads((WINDOW / name).read_text()), f'{row}: {name} differs from Nexar_window'
    windows = json.loads((d / constants.WINDOWS_FILENAME).read_text())
    forced = json.loads((d / constants.FORCED_NORMALS_FILENAME).read_text())
    assert {w: windows[w] for w in base_windows} == base_windows and set(windows) - set(base_windows) == set(forced), f'{row}: windows.json'
    assert len(forced) == info['edge_windows'] == len({windows[w]['source'] for w in forced}) <= 1689, (row, info)
    assert {windows[w]['source'] for w in forced} <= POS_IDS, f'{row}: a forced window from a non-positive train video'
    assert all(windows[w]['end'] - windows[w]['start'] == LENGTH for w in forced), f'{row}: a forced window is not {LENGTH} rows'
    print(f"{row}: {len(forced)} edge windows ({info['edge_windows_already_tiled']} already tiled, {len(info['edge_videos_without_room'])} videos without room) "
          f"= {len(forced) / 1689:.1%} of the normal slots | plan {plan}")


## 6. Train the rows of `RUN_ROWS` (§49 ED1, ED5, ED6)

**Watch the log: past 2 s/step, stop and paste it.** Then J10, the config diff against K5f (empty for the edge rows; exactly the two bottom-k keys for K5fbk), step-1 `mil` beside K5f's (K5fbk must match: same draws), last-epoch `mul_mil` above the §31 floor.


In [ ]:
K5_SETS = ['kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true', 'dvs.theta=1.0', 'dvs.theta_ego=1.0']
BASE = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
        f'data.dataset={DATASET}', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
        'kip.enabled=false', 'loss.dvs_anchor_mode=span', 'data.is_egocentric=false',
        f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}',
        f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}']
BK_KEYS = {'loss.bottomk_weight': constants.KIM_BOTTOMK_WEIGHT, 'loss.bottomk_topk_pct': constants.KIM_BOTTOMK_TOPK_PCT}
BK_SETS = [f'{k}={v}' for k, v in BK_KEYS.items()]
# row -> (sets, the config keys allowed to differ from K5f): edge rows differ by --data-dir only
ARMS = {PRE: (K5_SETS, {}), POST: (K5_SETS, {}), BK: ([*K5_SETS, *BK_SETS], BK_KEYS)}
DEFAULTS = _flat(Config().to_dict())


def config_diff(base, got):
    '''Keys that differ; a key added after ``base`` trained may sit at its default.'''
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out


def loss_print(rows, name):
    keys = ('mil', 'mul_mil', 'kim_mil', 'dvs_sup', 'bottomk')
    m = {k: sum(r.get(k, float('nan')) for r in rows) / len(rows) for k in keys}
    print(f'{name}: ' + ' | '.join(f'{k} {m[k]:.4f}' for k in keys))
    return m


def check_run(arm):
    _, allowed = ARMS[arm]
    base_cfg, base_run = k5f_cfg, K5F_RUN
    run_dir, ckpt, _ = row_paths(arm)
    payload = torch.load(ckpt, map_location='cpu', weights_only=False)
    assert payload['global_step'] == metrics_last(run_dir / 'metrics.jsonl')['global_step'] == STEPS, f'{arm}: J10'
    got = _flat(yaml.safe_load((run_dir / 'config.yaml').read_text()))
    diff = config_diff(base_cfg, got)
    assert {k: v[1] for k, v in diff.items()} == allowed, f'{arm}: config differs from K5f by {diff}, §49 allows {allowed}'
    rows = [json.loads(l) for l in (run_dir / 'metrics.jsonl').read_text().splitlines() if l.strip()]
    host = [json.loads(l) for l in (base_run / 'metrics.jsonl').read_text().splitlines() if l.strip()]
    first, h1 = loss_print(rows[:1], f'{arm} step 1'), loss_print(host[:1], 'K5f step 1')
    last = loss_print([r for r in rows if r['epoch'] == rows[-1]['epoch']], f'{arm} last epoch')
    loss_print([r for r in host if r['epoch'] == host[-1]['epoch']], 'K5f last epoch')
    same = abs(first['mil'] - h1['mil']) <= constants.KIM_START_TOL
    assert arm == BK or not same, f'{arm}: step-1 mil equals the one of K5f -- forced_normals.json was not read, this run IS K5f'
    note = ('OK' if same else 'FLAG: not the same draws -- paste to Claude') if arm == BK else 'forced items read: differs, as it must'
    print(f'ED6 step 1: mil {first["mil"]:.6f} vs K5f {h1["mil"]:.6f} ({note})')
    ok = last['mul_mil'] > constants.KIM_FIXED_MUL_MIL_FLOOR
    print(f'ED6 §31 check: last-epoch mul_mil {last["mul_mil"]:.4f} vs floor {constants.KIM_FIXED_MUL_MIL_FLOOR} -> '
          f'{"OK" if ok else "FLAG -- paste to Claude"}')


def train_arm(arm):
    sets, _ = ARMS[arm]
    run_dir, ckpt, _ = row_paths(arm)
    if is_verified(run_dir, STEPS):
        print(f'{arm} s{SEED}: verified on Drive -- not retrained')
        check_run(arm)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / arm
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in (*BASE, *sets) for x in ('--set', s)],
         '--data-dir', stage(DATA[arm], LOCAL / 'stage' / f'data_{arm}'), '--clip-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar'),
         '--output-dir', local, *resume], log=LOCAL / f'train_{arm}.log')
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{arm}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, run_dir / name)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{arm}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', run_dir / 'verified.json')
    print(f'{arm}: trained in {(time.time() - t0) / 60:.0f} min | mil {last.get("mil")} | kim_mil {last.get("kim_mil")}')
    check_run(arm)


for arm in RUN_ROWS:
    train_arm(arm)

## 7. Zero-shot reads of every row: DoTA-CAP-dev (protocol B), DADA (whole), TAD, DAD — K5f's protocols and bakes


In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_TAD = LOCAL / 'TAD_meta'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)


def read_evaluate(row, name, dataset, data_dir, bake, ids, out, ckpt):
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / row / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt, '--set', f'data.dataset={dataset}',
             '--data-dir', data_dir, '--clip-dir', stage(bake, LOCAL / 'stage' / f'{name}_bake', ids),
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'{row}_{name}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(ids), r
    print(f"{row} {name}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")


def read_zero_shot(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'dota' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt,
             '--input-dir', stage(A2_DOTA, LOCAL / 'stage' / 'cap', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels'), '--split', CAP_DEV,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_dota.log')
        publish(local, out)
    print(f"{row} DoTA-CAP-dev: macro {json.loads((out / 'protocol_b_readout.json').read_text())['macro']['mean']:.4f}")
    out = ev / 'dada' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', ckpt, '--t2-dir', L_T2,
             '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(A2_DADA, LOCAL / 'stage' / 'dada', DADA_IDS),
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{row}_dada.log')
        publish(local, out)
    print(f"{row} DADA whole: macro {json.loads((out / 'dada_readout.json').read_text())['macro']['mean']:.4f}")
    read_evaluate(row, 'tad', TAD_NAME, L_TAD, A2_TAD, TAD_IDS, ev / 'tad' / K5 / f's{SEED}', ckpt)
    read_evaluate(row, 'dad', DAD_NAME, stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), A2_DAD, DAD_IDS,
                  ev / constants.DAD_OUTPUT_SET / K5 / f's{SEED}', ckpt)
    flush_drive()


for row in RUN_ROWS:
    read_zero_shot(row)

## 8. `nexar_val` read of every row with `--save-scores` (`--final` is never passed)


In [ ]:
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON, VIDEO_STATS_JSON  # noqa: E402


def read_nexar(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'nexar' / row
    done = all((out / f).exists() for f in (READOUT_JSON, ITEM_AUCS_JSON, VIDEO_STATS_JSON)) and \
        len(list((out / 'scores').glob('*.npz'))) == len(NEXAR_IDS)
    if not done:
        L_X = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
        L_U = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
        L_STATS = stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', [])     # manifest + statistics only
        local = LOCAL / 'eval' / row / 'nexar'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', ckpt, '--stats-dir', L_STATS,
             '--video-s1-dir', L_U, '--census', CENSUS, '--clip-s1-dir', L_X,
             '--data-dir', L_DEFS, '--split', NEXAR_SPLIT, '--save-scores', '--out-dir', local], log=LOCAL / f'{row}_nexar.log')
        publish(local, out)
        flush_drive()
    r = json.loads((out / READOUT_JSON).read_text())
    assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS), (r['split'], r['videos'])
    assert r['global_steps'] == {f's{SEED}': STEPS} and Path(r['checkpoints'][f's{SEED}']) == ckpt, (r['global_steps'], r['checkpoints'])
    print(f"{row}: crops macro {r['crops']['mean_over_placements']['macro']['mean']:.4f} | "
          f"whole macro {r['whole']['macro']['mean']:.4f} | clip AUC {r['whole']['clip_auc_max']:.4f}")


for row in RUN_ROWS:
    read_nexar(row)

## 9. Rows vs K5f -> **paste `REPORTS/k5_edge_summary.md` to Claude**

Reports every row complete on Drive. `k5_edge_report` (§49 ED8–ED11: T1, T2, labels, verdicts), `zs_arm_table` (control K5f: macro **and** micro on the four zero-shot sets), `pre_onset_read` on Nexar val, DoTA-CAP-dev and DADA (printed). **K5f stays the reference.**


In [ ]:
def complete(row):
    run_dir, _, ev = row_paths(row)
    return is_verified(run_dir, STEPS) and all(p.exists() for p in (
        ev / 'dota' / K5 / 'clip_aucs.json', ev / 'dada' / K5 / 'clip_aucs.json', ev / 'tad' / K5 / f's{SEED}' / 'results.json',
        ev / constants.DAD_OUTPUT_SET / K5 / f's{SEED}' / 'results.json', ev / 'nexar' / row / READOUT_JSON))


DONE = [row for row in (PRE, POST, BK) if complete(row)]
assert DONE, 'no row is complete'
print('complete rows:', DONE)
local = LOCAL / 'report'
shutil.rmtree(local, ignore_errors=True)
local.mkdir(parents=True)
run([sys.executable, '-m', 'core.tools.k5_edge_report', '--kim-root', FIXED / K5F, '--k5f-nexar-dir', K5F_EV / 'nexar' / K5F,
     '--edge-root', ROOT, '--rows', *DONE, '--out-dir', local])
reads = []
for arm, ev in ((K5F, K5F_EV), *((row, row_paths(row)[2]) for row in DONE)):
    for s in ('dota', 'dada', 'tad', constants.DAD_OUTPUT_SET):
        reads += ['--read', s, arm, ev / s / K5]
NAME = constants.KIM_EDGE_TABLE_NAME
run([sys.executable, '-m', 'core.tools.zs_arm_table', '--control', K5F, '--seed', SEED, *reads, '--name', NAME, '--out-dir', local])
PO = local / constants.PRE_ONSET_DIRNAME
for s in ('nexar', 'dota', 'dada'):
    rows = []
    for arm, ev in ((K5F, K5F_EV), *((row, row_paths(row)[2]) for row in DONE)):
        rows += ['--row', f"{arm}={ev / 'nexar' / arm if s == 'nexar' else ev / s / K5 / 'clip_scores.npz'}"]
    run([sys.executable, '-m', 'core.tools.pre_onset_read', '--set', s, '--control', K5F, *rows, '--out-dir', PO, '--plots'])
parts = [(local / f'{constants.KIM_EDGE_REPORT_STEM}.md').read_text(),
         '# Four sets, macro and micro\n\n' + (local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md').read_text()]
for s in ('nexar', 'dota', 'dada'):
    stem = constants.PRE_ONSET_REPORT_STEM + ('' if s == 'nexar' else f'_{s}')
    parts.append((PO / f'{stem}.md').read_text())
(local / 'k5_edge_summary.md').write_text('\n\n'.join(parts))
publish(local, REPORTS)
print((REPORTS / 'k5_edge_summary.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5_edge_summary.md to Claude')
